In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

from causal_opt.fmri_data import (
    load_fmri_state_data,
    load_paired_tsv_states,
)

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
FMRI_ROOT = Path(
    os.getenv(
        "FMRI_CONNECTIVITY_ROOT",
        ROOT.parent / "fmri_connectivity",
    )
)

# Original CONN BN19 outputs
CONN_ROOT = (
    FMRI_ROOT
    / "data"
    / "mat_files"
    / "rs_sessions_r03_healthy"
)

CONN_SESSION1 = CONN_ROOT / "roi_rs_sessions_Session1.zip"
CONN_SESSION2 = CONN_ROOT / "roi_rs_sessions_Session2.zip"

# New fMRIPrep-derived BN19 outputs
BN19_TS_ROOT = (
    FMRI_ROOT
    / "data"
    / "derivatives"
    / "r03_healthy"
    / "rois"
    / "BN19"
)

BN19_LABELS = (
    FMRI_ROOT
    / "data"
    / "templates"
    / "ROIs"
    / "BN19_labels.json"
)

# ---------------------------------------------------------
# Load both pipelines
# ---------------------------------------------------------
conn_control, conn_sdv = load_fmri_state_data(
    CONN_SESSION1,
    CONN_SESSION2,
)

fmriprep_control, fmriprep_sdv = load_paired_tsv_states(
    BN19_TS_ROOT,
    BN19_LABELS,
    pairing="intersection",
)

# ---------------------------------------------------------
# Match BN19 ROIs using MNI coordinates
# ---------------------------------------------------------
D = np.linalg.norm(
    conn_control.roi_xyz[:, None, :]
    - fmriprep_control.roi_xyz[None, :, :],
    axis=2,
)

conn_idx, prep_idx = linear_sum_assignment(D)

# Put matching fMRIPrep columns into CONN order
order = prep_idx[np.argsort(conn_idx)]

matched_distances = D[conn_idx, prep_idx]

print(
    "ROI coordinate matching:"
    f" median={np.median(matched_distances):.3f} mm,"
    f" max={np.max(matched_distances):.3f} mm"
)

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def zscore_columns(X):
    X = np.asarray(X, float)
    std = X.std(axis=0, ddof=1)
    return (X - X.mean(axis=0)) / std


def align_with_lag(A, B, lag):
    """
    Compare CONN A[t] with fMRIPrep B[t + lag].
    Positive lag = fMRIPrep shifted later.
    """
    if lag >= 0:
        n = min(len(A), len(B) - lag)
        return A[:n], B[lag:lag+n]
    else:
        lag = -lag
        n = min(len(A) - lag, len(B))
        return A[lag:lag+n], B[:n]


def compare_subject_state(conn_state, prep_state, subject, max_lag=5):

    A = np.asarray(conn_state.timeseries[subject], float)
    B = np.asarray(prep_state.timeseries[subject], float)[:, order]

    print(
        f"{subject}: CONN shape={A.shape}, "
        f"fMRIPrep shape={B.shape}"
    )

    # Search a small lag range in case dummy-volume handling differs
    lag_scores = {}

    for lag in range(-max_lag, max_lag + 1):
        a, b = align_with_lag(A, B, lag)

        za = zscore_columns(a)
        zb = zscore_columns(b)

        roi_r = np.array([
            np.corrcoef(za[:, j], zb[:, j])[0, 1]
            for j in range(za.shape[1])
        ])

        lag_scores[lag] = np.median(roi_r)

    best_lag = max(lag_scores, key=lag_scores.get)

    A2, B2 = align_with_lag(A, B, best_lag)
    ZA = zscore_columns(A2)
    ZB = zscore_columns(B2)

    roi_r = np.array([
        np.corrcoef(ZA[:, j], ZB[:, j])[0, 1]
        for j in range(ZA.shape[1])
    ])

    # Compare functional-connectivity structure too
    FC_A = np.corrcoef(ZA, rowvar=False)
    FC_B = np.corrcoef(ZB, rowvar=False)

    upper = np.triu_indices_from(FC_A, k=1)

    fc_r = np.corrcoef(
        FC_A[upper],
        FC_B[upper],
    )[0, 1]

    # Raw amplitude difference
    std_ratio = np.median(
        B2.std(axis=0, ddof=1)
        / A2.std(axis=0, ddof=1)
    )

    roi_table = pd.DataFrame({
        "CONN ROI": np.asarray(conn_state.roi_names),
        "fMRIPrep ROI": np.asarray(prep_state.roi_names)[order],
        "MNI distance mm": matched_distances,
        "timeseries r": roi_r,
        "CONN std": A2.std(axis=0, ddof=1),
        "fMRIPrep std": B2.std(axis=0, ddof=1),
    })

    summary = {
        "subject": subject,
        "best_lag_TR": best_lag,
        "median_ROI_r": np.median(roi_r),
        "min_ROI_r": np.min(roi_r),
        "max_ROI_r": np.max(roi_r),
        "FC_matrix_r": fc_r,
        "median_std_ratio_fmriprep_over_CONN": std_ratio,
    }

    return summary, roi_table, lag_scores


# ---------------------------------------------------------
# First quick check: Subject002 control
# ---------------------------------------------------------
summary, roi_table, lag_scores = compare_subject_state(
    conn_control,
    fmriprep_control,
    "Subject002",
)

display(pd.Series(summary))
display(
    roi_table.sort_values("timeseries r")
)

print("Lag search:")
display(pd.Series(lag_scores, name="median ROI correlation"))